In [ ]:
!pip install ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.3 MB/s eta 0:00:00


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from google.colab import files
uploaded=files.upload()

for fn in uploaded.keys():
  print(f'fichier "{fn}" importe avec succes')

Saving dataset_phase1_final_READY.jsonl to dataset_phase1_final_READY.jsonl
fichier "dataset_phase1_final_READY.jsonl" importe avec succes


In [ ]:
!pip install rouge_score evaluate absl-py


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=22ad5cd5304972d28778c18336cc69b3325a6b82f2fc40ed3bdb08203948218f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from huggingface_hub import notebook_login
notebook_login()



In [ ]:
import os
import random
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
    GenerationConfig
)
from transformers.trainer_utils import get_last_checkpoint
from huggingface_hub import snapshot_download

# =========================================================
# 1. SEED (REPRODUCTIBILITÉ SCIENTIFIQUE)
# =========================================================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# =========================================================
# 2. CONFIGURATION
# =========================================================
model_id = "Fatoumataa/mt5-bambara-phase0-pro"
dataset_path = "/content/dataset_phase1_final_READY.jsonl"
repo_id_final = "Fatoumataa/mt5-bambara-resumer-final"
output_dir = "./mt5-bambara-resumer-final"

# =========================================================
# 3. CHARGEMENT MODÈLE + TOKENIZER
# =========================================================
print("---- Chargement modèle Phase 0 ----")
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# =========================================================
# 4. CONFIGURATION DE GÉNÉRATION (PROPRE)
# =========================================================
generation_config = GenerationConfig(
    max_length=128,
    min_length=10,
    num_beams=4,
    length_penalty=1.2,
    no_repeat_ngram_size=3,
    repetition_penalty=2.5,
    decoder_start_token_id=tokenizer.pad_token_id,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

model.generation_config = generation_config

# =========================================================
# 5. CHARGEMENT DATASET
# =========================================================
print("---- Chargement dataset ----")
df = pd.read_json(dataset_path, lines=True)
df["id"] = df["id"].astype(str)

full_ds = Dataset.from_pandas(df)
dataset = full_ds.train_test_split(test_size=0.1, seed=seed)

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=512,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=128,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# =========================================================
# 6. MÉTRIQUE ROUGE SÉCURISÉE
# =========================================================
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    vocab_size = tokenizer.vocab_size

    preds = np.where(
        (preds >= 0) & (preds < vocab_size),
        preds,
        tokenizer.pad_token_id,
    )

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_preds = [p.strip() for p in decoded_preds]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.where(
        (labels >= 0) & (labels < vocab_size),
        labels,
        tokenizer.pad_token_id,
    )

    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False
    )

    return {k: round(v, 4) for k, v in result.items()}

# =========================================================
# 7. TRAINING ARGUMENTS
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    # --- MODIFICATIONS STRATÉGIQUES ---
    learning_rate=1e-5,               # Réduit (de 2e-5 à 1e-5) pour éviter l'instabilité
    weight_decay=0.05,                # Augmenté pour forcer le modèle à ne pas sur-apprendre
    warmup_steps=0,                   # On retire le warmup pour commencer direct avec Adafactor
    lr_scheduler_type="constant",     # On garde un taux stable pour éviter les sauts brusques

    # LE POINT CRUCIAL : Désactivation du FP16
    fp16=False,                       # FORCE à False. Le FP16 est souvent la cause des Loss: nan
    # ----------------------------------

    label_smoothing_factor=0.1,
    num_train_epochs=8,
    optim="adafactor",                # Très bon pour mT5, il va gérer seul l'échelle des gradients
    gradient_checkpointing=True,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    predict_with_generate=True,
    generation_config=generation_config,
    push_to_hub=True,
    hub_model_id=repo_id_final,
    hub_strategy="checkpoint",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    seed=seed,
    data_seed=seed,
    report_to="none"
)

# =========================================================
# 8. INITIALISATION TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# =========================================================
# 9. GESTION CHECKPOINT ROBUSTE
# =========================================================
last_checkpoint = None

if os.path.isdir(output_dir):
    last_checkpoint = get_last_checkpoint(output_dir)

if last_checkpoint is not None:
    print(f"---- Reprise depuis checkpoint local : {last_checkpoint} ----")
else:
    print("---- Aucun checkpoint local trouvé. Nouvel entraînement ----")

# =========================================================
# 10. LANCEMENT
# =========================================================
print("🚀 Lancement du Fine-tuning Phase 1")
trainer.train(resume_from_checkpoint=last_checkpoint)

# =========================================================
# 11. PUSH FINAL
# =========================================================
print("---- Upload final ----")
trainer.push_to_hub(
    commit_message="Phase 1 finale - Résumé bambara stabilisé"
)

---- Chargement modèle Phase 0 ----


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


---- Chargement dataset ----


Map:   0%|          | 0/18040 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,35.304016,3.835557,0.456200,0.215100,0.305700,0.305600
2,32.929055,3.696899,0.475600,0.230400,0.323700,0.323900
3,31.617004,3.589498,0.484900,0.237200,0.331400,0.331400


KeyboardInterrupt: 

In [ ]:
from huggingface_hub import snapshot_download

# On télécharge ce qui est sur le Hub vers ton dossier local
snapshot_download(
    repo_id="Fatoumataa/mt5-bambara-resumer-boost1",
    local_dir="./mt5-bambara-resumer-boost1",
    repo_type="model"
)

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

'/content/mt5-bambara-resumer-boost1'

In [ ]:
import os
import random
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
    GenerationConfig
)
from huggingface_hub import snapshot_download

# 1. SEED (Strictement identique pour l'ordre des données)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# 2. CONFIGURATION
model_id = "Fatoumataa/mt5-bambara-phase0-pro"
dataset_path = "/content/dataset_phase1_final_READY.jsonl"
repo_id_final = "Fatoumataa/mt5-bambara-resumer-final"
output_dir = "./mt5-bambara-resumer-final"

# 3. TÉLÉCHARGEMENT
print("---- Téléchargement du checkpoint depuis le Hub ----")
snapshot_download(repo_id=repo_id_final, local_dir=output_dir, repo_type="model")

# 4. CHARGEMENT MODÈLE + TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# 5. CONFIGURATION DE GÉNÉRATION (Exactement ton code initial)
generation_config = GenerationConfig(
    max_length=128, min_length=10, num_beams=4, length_penalty=1.2,
    no_repeat_ngram_size=3, repetition_penalty=2.5,
    decoder_start_token_id=tokenizer.pad_token_id,
    pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
)
model.generation_config = generation_config

# 6. CHARGEMENT DATASET (Correction technique id incluse)
df = pd.read_json(dataset_path, lines=True)
df["id"] = df["id"].astype(str)
full_ds = Dataset.from_pandas(df)
dataset = full_ds.train_test_split(test_size=0.1, seed=seed)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True, padding=False)
    labels = tokenizer(text_target=examples["target_text"], max_length=128, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

# 7. MÉTRIQUE ROUGE (Exactement ton code initial)
rouge_metric = evaluate.load("rouge")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    vocab_size = tokenizer.vocab_size
    preds = np.where((preds >= 0) & (preds < vocab_size), preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.where((labels >= 0) & (labels < vocab_size), labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {k: round(v, 4) for k, v in result.items()}

# 8. TRAINING ARGUMENTS (COPIE CONFORME DE TON CODE INITIAL)
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,             # Remis à 50 pour correspondre au checkpoint
    learning_rate=1e-5,
    weight_decay=0.05,
    warmup_steps=0,
    lr_scheduler_type="constant",
    fp16=False,                   # Gardé à False pour éviter les NaN
    label_smoothing_factor=0.1,
    num_train_epochs=8,
    optim="adafactor",
    gradient_checkpointing=False, # Mis à False pour éviter l'erreur de recomputation
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    predict_with_generate=True,
    generation_config=generation_config,
    push_to_hub=True,
    hub_model_id=repo_id_final,
    hub_strategy="checkpoint",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    seed=seed,
    data_seed=seed,
    report_to="none"
)

# 9. GESTION CHECKPOINT
last_checkpoint = os.path.join(output_dir, "last-checkpoint")
if os.path.exists(os.path.join(last_checkpoint, "trainer_state.json")):
    print(f"✅ REPRISE DÉTECTÉE : {last_checkpoint}")
    # Nettoyage automatique du fichier de conflit
    if os.path.exists(os.path.join(last_checkpoint, "scaler.pt")):
        os.remove(os.path.join(last_checkpoint, "scaler.pt"))
else:
    last_checkpoint = None

# 10. INITIALISATION + LANCEMENT
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Lancement du Fine-tuning (Reprise exacte)")
trainer.train(resume_from_checkpoint=last_checkpoint)

# 11. PUSH FINAL
trainer.push_to_hub(commit_message="Phase 1 - Reprise réussie vers Epoch 8")

---- Téléchargement du checkpoint depuis le Hub ----


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/18040 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
4,30.451165,3.498846,0.486300,0.238800,0.334300,0.334400
5,29.923555,3.443114,0.491500,0.244300,0.340200,0.340300
6,29.376394,3.396045,0.502200,0.251300,0.345900,0.345900
7,29.022422,3.352131,0.507200,0.255300,0.351700,0.351700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

# On vérifie où on en était
last_checkpoint = get_last_checkpoint("./mt5-bambara-resumer-final")

if last_checkpoint:
    print(f"✅ Checkpoint trouvé : {last_checkpoint}. Reprise en cours...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("⚠️ Aucun checkpoint trouvé. Vérifie le chemin du dossier !")

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================================================
# 1. CONFIGURATION ET DÉTECTION DU CHECKPOINT
# =========================================================
model_id = "Fatoumataa/mt5-bambara-phase0-pro"
output_dir = "./mt5-bambara-resumer-final"

# Détection automatique du dernier checkpoint local
checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
if not checkpoints:
    raise ValueError(f"Aucun dossier checkpoint trouvé dans {output_dir}")

# On prend le numéro le plus élevé (le plus récent)
last_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
checkpoint_path = os.path.abspath(os.path.join(output_dir, last_checkpoint))

print(f"🚀 Modèle chargé depuis : {last_checkpoint}")

# =========================================================
# 2. CHARGEMENT DU MODÈLE ET DU TOKENIZER
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint_path,
    local_files_only=True
).to("cuda")

# =========================================================
# 3. FONCTION DE GÉNÉRATION (INFÉRENCE)
# =========================================================
def tester_resume(texte_entree):
    # Préparation de l'entrée
    inputs = tokenizer(
        texte_entree,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to("cuda")

    # Paramètres de génération optimisés
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,      # Longueur max du résumé
            min_length=10,       # Forcer une phrase complète
            num_beams=4,         # Recherche par faisceau pour la qualité
            repetition_penalty=2.5,
            length_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================================================
# 4. BATTERIE DE TESTS (8 EXEMPLES VARIÉS)
# =========================================================
exemples_test = [
    "Moussa bɛ kàlanso la, a bɛ kalan kɛ kosɔbɛ k'a sɔrɔ a bɛ baara kɛ u ka nakɔ la.",
    "Sini mɔgɔw bɛna bɛn sènɛko kan, kaba ni tìga bɛna kɛ kosɔbɛ san fɛ n'a tìlè nàna.",
    "A bɛ fɔ ko mɔgɔw ka kan ka u bolo ko ni safìnɛ ye tuma bɛɛ kà sɔrɔ u bɛ balo munu.",
    "Suku dɔgɔkun in na, mɔgɔw ma sɔrɔ ka tìga sòngò dɔgɔya, o y'a to mɔgɔ fanba ma tìga sɔrɔ.",
    "Muso dɔ bɛ sìnɛ fɛ, a bɛ nòno firi k'a sìn dènw ma, o ye dènw ka kɛnɛya ko ye.",
    "Jiradenw bɛ bèn kosɔbɛ n'a tìlè ma jà, bari n'a tìlè jàra, jiriw bɛ jà kà sa.",
    "Cɛ̀ in bɛ baara kɛ k'a sɔrɔ a tɛ dɛsɛ, o de y'a to a sɔrɔ la kà fɛn caman sɔrɔ a ka baara fɛ.",
    "Kuma bɛ tàga mɔgɔ caman ma ni tèlèfɔni ye, o bɛ mɔgɔw kònɔ bɛn dugu kònɔ."
]

print("\n" + "="*50)
print(f"{'TEXTE ORIGINAL':<60} | {'RÉSUMÉ GÉNÉRÉ'}")
print("="*50)

for i, text in enumerate(exemples_test, 1):
    res = tester_resume(text)
    # Formatage pour une lecture propre
    original_court = (text[:57] + '...') if len(text) > 60 else text
    print(f"{i}. {original_court:<57} | {res}")

print("="*50)

🚀 Modèle chargé depuis : checkpoint-3384


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



TEXTE ORIGINAL                                               | RÉSUMÉ GÉNÉRÉ
1. Moussa bɛ kàlanso la, a bɛ kalan kɛ kosɔbɛ k'a sɔrɔ a bɛ ... | Moussa bɛ kalan kɛ kosɔbɛ k'a sɔrɔ a ka nakɔ la.
2. Sini mɔgɔw bɛna bɛn sènɛko kan, kaba ni tìga bɛna kɛ kosɔ... | Sini mɔgɔw bɛna bɛn sènɛko kan, kaba ni tìga bɛ na kɛ kosɔbɛ san fɛ n'a tìlè nàna.
3. A bɛ fɔ ko mɔgɔw ka kan ka u bolo ko ni safìnɛ ye tuma bɛ... | bɛ fɔ ko mɔgɔw ka kan ka u bolo ko ni safìnɛ ye tuma bɛɛ kà sɔrɔ u bɛ balo munu.
4. Suku dɔgɔkun in na, mɔgɔw ma sɔrɔ ka tìga sòngò dɔgɔya, o... | fini na, mɔgɔw ma sɔrɔ ka tìga sòngò dɔgɔnya, o y'a to mɔgo fanba ma n'a k'a fɔmɔgɛ fanba in na.
5. Muso dɔ bɛ sìnɛ fɛ, a bɛ nòno firi k'a sìn dènw ma, o ye ... | ɛ fɛ, a bɛ nòno firi k'a sìn dènw ma, o ye ɲɔrɔ ka kɛnɛya ko ye. Muso dɔ bɛ mɔgɔ
6. Jiradenw bɛ bèn kosɔbɛ n'a tìlè ma jà, bari n'a tìlè jàra... | <extra_id_0> jiriw bɛ bèn kosɔbɛ n'a tìlè ma jà, bari kà sa.
7. Cɛ̀ in bɛ baara kɛ k'a sɔrɔ a tɛ dɛsɛ, o de y'a to a sɔrɔ... | in bɛ ba

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================================================
# 1. CONFIGURATION ET DÉTECTION DU CHECKPOINT
# =========================================================
model_id = "Fatoumataa/mt5-bambara-phase0-pro"
output_dir = "./mt5-bambara-resumer-final"

# Détection automatique du dernier checkpoint local
checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
if not checkpoints:
    raise ValueError(f"Aucun dossier checkpoint trouvé dans {output_dir}")

last_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
checkpoint_path = os.path.abspath(os.path.join(output_dir, last_checkpoint))

print(f"🚀 Modèle chargé depuis : {last_checkpoint}")

# =========================================================
# 2. CHARGEMENT DU MODÈLE ET DU TOKENIZER
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint_path,
    local_files_only=True
).to("cuda")

# =========================================================
# 3. FONCTION DE GÉNÉRATION "PRO" (ANTI-BUGS)
# =========================================================
def tester_resume_strict(texte_entree):
    # On retire le prefixe "summarize:" s'il n'était pas dans ton dataset
    inputs = tokenizer(
        texte_entree,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=50,      # On limite la longueur pour éviter les dérives
            num_beams=2,            # On baisse le beam pour éviter que le modèle "cherche trop loin"
            repetition_penalty=1.2, # On baisse encore la pénalité (trop de pénalité tue la grammaire)
            do_sample=False,        # On désactive l'aléatoire
            early_stopping=True,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.strip()

# =========================================================
# 4. BATTERIE DE TESTS (8 EXEMPLES)
# =========================================================
exemples_test = [
    "Moussa bɛ kàlanso la, a bɛ kalan kɛ kosɔbɛ k'a sɔrô a bɛ baara kɛ u ka nakɔ la.",
    "Sini mɔgɔw bɛna bɛn sènɛko kan, kaba ni tìga bɛna kɛ kosɔbɛ san fɛ n'a tìlè nàna.",
    "A bɛ fɔ ko mɔgɔw ka kan ka u bolo ko ni safìnɛ ye tuma bɛɛ kà sɔrɔ u bɛ balo munu.",
    "Suku dɔgɔkun in na, mɔgɔw ma sɔrɔ ka tìga sòngò dɔgɔya, o y'a to mɔgɔ fanba ma tìga sɔrɔ.",
    "Muso dɔ bɛ sìnɛ fɛ, a bɛ nòno firi k'a sìn dènw ma, o ye dènw ka kɛnɛya ko ye.",
    "Jiradenw bɛ bèn kosɔbɛ n'a tìlè ma jà, bari n'a tìlè jàra, jiriw bɛ jà kà sa.",
    "Cɛ̀ in bɛ baara kɛ k'a sɔrɔ a tɛ dɛsɛ, o de y'a to a sɔrɔ la kà fɛn caman sɔrɔ a ka baara fɛ.",
    "Kuma bɛ tàga mɔgɔ caman ma ni tèlèfɔni ye, o bɛ mɔgɔw kònɔ bɛn dugu kònɔ."
]

print("\n" + "="*80)
print(f"{'TEXTE ORIGINAL':<65} | {'RÉSUMÉ PRO'}")
print("="*80)

for i, text in enumerate(exemples_test, 1):
    res = tester_resume_pro(text)
    # Affichage tronqué pour la lisibilité
    original_court = (text[:62] + '...') if len(text) > 65 else text
    print(f"{i}. {original_court:<62} | {res}")

print("="*80)

🚀 Modèle chargé depuis : checkpoint-3384


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



TEXTE ORIGINAL                                                    | RÉSUMÉ PRO
1. Moussa bɛ kàlanso la, a bɛ kalan kɛ kosɔbɛ k'a sɔrô a bɛ baara... | fini k'a sɔrô a bɛ kalan kɛ kosɔbɛ k'u ka nakɔ la. Moussa ka kàlanso la, o bɛ baara kɛ u ka dɔrɔ la,
2. Sini mɔgɔw bɛna bɛn sènɛko kan, kaba ni tìga bɛna kɛ kosɔbɛ sa... | fini mɔgɔw bɛna bɛn sènɛko kan, kaba ni tìga bɛ na kɛ kosɔbɛ san fɛ n'a tìlè nàna.
3. A bɛ fɔ ko mɔgɔw ka kan ka u bolo ko ni safìnɛ ye tuma bɛɛ kà ... | bɛ fɔ ko mɔgɔw ka kan ka u bolo ko ni safìnɛ ye tuma bɛɛ kà sɔrɔ u bɛ balo munu. A bɛfɔ ko ɲɔgɔnw ka kunna ka a bɛ bɔ ko ni sigininɛ ye tun bɛɛnɛ kɛ k'u
4. Suku dɔgɔkun in na, mɔgɔw ma sɔrɔ ka tìga sòngò dɔgɔya, o y'a ... | fini na, mɔgɔw ma sɔrɔ ka tìga sòngò dɔgɔnya, o y'a to mɛgɔ fanba ma n'a k'u to mɔgo fanba ka tɔga jɔ
5. Muso dɔ bɛ sìnɛ fɛ, a bɛ nòno firi k'a sìn dènw ma, o ye dènw ... | kupuo dɔ bɛ sìnɛ fɛ, a bɛ kɛnɛya ko ye.
6. Jiradenw bɛ bèn kosɔbɛ n'a tìlè ma jà, bari n'a tìlè jàra, jir... | jiriw bɛ bèn ko